In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 시간-주파수 분석 (Short-Time Fourier Transform)

FFT 는 신호 전체를 하나의 스펙트럼으로 압축하기 때문에 **언제 어떤 주파수가 나타났는지**를 알려주지 않는다. 엔진음, 타건음, 충격성 이벤트처럼 시간에 따라 성분이 변하는 신호를 다루려면 짧은 구간을 잘라 각 구간별로 FFT 를 수행하는 STFT 가 필요하다.

이 노트북에서는 세 가지 엔진 관련 음원 (`autorash`, `knocking`, `f1music`) 을 사용해

- `nperseg` (세그먼트 길이) 가 시간/주파수 해상도 트레이드오프를 어떻게 결정하는지
- `noverlap` (세그먼트 중첩) 이 시간축 부드러움을 어떻게 좌우하는지
- Mel 스케일과 dB 변환이 청각적 해석에 주는 효과

를 차례로 관찰한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import utils
from scipy.signal import stft

try:
    import librosa
    import librosa.display
except ImportError:
    %pip install librosa
    import librosa
    import librosa.display

---


## 실습 1. 두 엔진음 로드 및 시간/주파수 일별

- `autorash.mp3` : 가솔린 엔진 공회전음 (정상 상태)
- `knocking.mp3` : 엔진 노킹 (이상 연소로 인한 주기적 충격)

두 신호의 샘플링 주파수가 같도록 로드한 뒤 길이를 맞추고, 시간축 파형과 FFT 진폭 스펙트럼을 2 × 2 subplot 으로 나란히 비교한다. 시간축 생성에는 부동소수점 누적 오차가 없는 `np.arange(len(v))/fs` 를 사용한다.

In [ ]:
# 데이터 로드 (원본 샘플링 그대로)
v_a, fs_a = librosa.load('./data/autorash.mp3',  sr=None)
v_k, fs_k = librosa.load('./data/knocking.mp3', sr=None)

# 두 신호 길이를 짧은 쪽 (knocking) 에 맞춤
N = min(len(v_a), len(v_k))
v_a = v_a[:N]
v_k = v_k[:N]

# 시간축 — 부동소수점 누적 오차 방지 위해 index / fs 방식
t_a_raw = np.arange(len(v_a)) / fs_a
t_k_raw = np.arange(len(v_k)) / fs_k

print(f'Autorash : shape={v_a.shape}, fs={fs_a} Hz, duration={len(v_a)/fs_a:.2f} s')
print(f'Knocking : shape={v_k.shape}, fs={fs_k} Hz, duration={len(v_k)/fs_k:.2f} s')

# 주파수 스펙트럼 (단측)
f_a, A_a = utils.fft(v_a, fs_a)
f_k, A_k = utils.fft(v_k, fs_k)

fig, ax = plt.subplots(2, 2, figsize=(12, 6))

ax[0, 0].plot(t_a_raw, v_a, 'C0')
ax[0, 0].set_title('Autorash — Time')
ax[0, 0].set_xlabel('Time (s)'); ax[0, 0].set_ylabel('Amplitude')

ax[0, 1].plot(f_a, A_a, 'C0')
ax[0, 1].set_title('Autorash — Spectrum')
ax[0, 1].set_xlabel('Frequency (Hz)'); ax[0, 1].set_ylabel('Magnitude')
ax[0, 1].set_xlim(0, 2000)

ax[1, 0].plot(t_k_raw, v_k, 'C1')
ax[1, 0].set_title('Knocking — Time')
ax[1, 0].set_xlabel('Time (s)'); ax[1, 0].set_ylabel('Amplitude')

ax[1, 1].plot(f_k, A_k, 'C1')
ax[1, 1].set_title('Knocking — Spectrum')
ax[1, 1].set_xlabel('Frequency (Hz)'); ax[1, 1].set_ylabel('Magnitude')
ax[1, 1].set_xlim(0, 2000)

fig.tight_layout(); plt.show()

**관찰.** 시간축 파형만 보면 두 신호 모두 잡음처럼 보이지만, `knocking` 쪽은 일정 간격의 충격성 스파이크가 육안으로도 식별된다. 반대로 주파수 스펙트럼만 보면 하모닉 피크 몇 개가 보일 뿐 **충격이 언제 발생했는지**는 전혀 알 수 없다. 이 정보 손실을 메우는 도구가 바로 STFT 다.

---


## 실습 2. 첫 스펙트로그램 (`nperseg=256`, `noverlap=128`)

STFT 는 신호를 길이 `nperseg` 의 세그먼트로 쪼개 각 세그먼트에 FFT 를 적용한다. 세그먼트 간 `noverlap` 샘플만큼 겹치면서 시간 방향으로 이동한다.

- `nperseg = 256` → 세그먼트 길이 `256 / 44100 ≈ 5.8 ms` : 충격성 이벤트를 시간적으로 분해하기 좋다.
- `noverlap = 128` → 50 % 중첩.
- 주파수 해상도는 `fs / nperseg = 44100 / 256 ≈ 172 Hz` 로 비교적 거칠다.

두 엔진음에 대해 스펙트로그램을 그리되, STFT 의 반환 시간축은 원 시간축과 섞이지 않도록 `t_a_stft`, `t_k_stft` 라는 **별도 변수**에 저장한다.

In [ ]:
nperseg  = 256
noverlap = 128
nfft     = 256

f_a_stft, t_a_stft, Zxx_a = stft(v_a, fs=fs_a,
                                  nperseg=nperseg, noverlap=noverlap, nfft=nfft)
f_k_stft, t_k_stft, Zxx_k = stft(v_k, fs=fs_k,
                                  nperseg=nperseg, noverlap=noverlap, nfft=nfft)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

im0 = ax[0].pcolormesh(t_a_stft, f_a_stft, np.abs(Zxx_a),
                        shading='gouraud', cmap='viridis')
ax[0].set_title(f'Autorash — STFT (nperseg={nperseg})')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Frequency (Hz)')
ax[0].set_ylim(0, 10000)
fig.colorbar(im0, ax=ax[0], label='Magnitude')

im1 = ax[1].pcolormesh(t_k_stft, f_k_stft, np.abs(Zxx_k),
                        shading='gouraud', cmap='viridis')
ax[1].set_title(f'Knocking — STFT (nperseg={nperseg})')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Frequency (Hz)')
ax[1].set_ylim(0, 10000)
fig.colorbar(im1, ax=ax[1], label='Magnitude')

fig.tight_layout(); plt.show()

**관찰.** `autorash` 는 가로 줄무늬 (지속 하모닉) 가 시간에 걸쳐 평탄하게 이어지는 반면, `knocking` 은 세로 줄무늬 (짧은 광대역 폭발) 가 주기적으로 반복된다. FFT 한 장으로는 전혀 보이지 않던 **시간적 구조**가 드러났다는 점이 중요하다. 다만 주파수 해상도가 170 Hz 수준으로 거칠어 하모닉 간 간격이 좁은 영역은 뭉개진다.

---


## 실습 3. `nperseg` 비교 — 시간/주파수 해상도 트레이드오프

세그먼트 길이를 바꾸면 시간 해상도와 주파수 해상도가 서로 반대로 움직인다 (시간-주파수 불확정성). `knocking` 신호에 대해 세 가지 값을 나란히 비교한다.

| `nperseg` | 세그먼트 길이 | 주파수 해상도 `fs/nperseg` | 특징 |
|-----------|--------------|----------------------------|------|
| 256       | ≈ 5.8 ms     | ≈ 172 Hz                   | 충격 이벤트 시점 뚜렷, 주파수 뭉개짐 |
| 2048      | ≈ 46 ms      | ≈ 22 Hz                    | 균형형 |
| 8192      | ≈ 186 ms     | ≈ 5.4 Hz                   | 하모닉 선명, 시간 변동 흐릿 |

중첩 비율은 모두 50 % 로 고정한다.

In [ ]:
nperseg_list = [256, 2048, 8192]

fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, nps in enumerate(nperseg_list):
    f_k_i, t_k_i, Zxx_k_i = stft(v_k, fs=fs_k,
                                  nperseg=nps, noverlap=nps // 2, nfft=nps)
    im = ax[i].pcolormesh(t_k_i, f_k_i, np.abs(Zxx_k_i),
                           shading='gouraud', cmap='viridis')
    ax[i].set_title(f'nperseg={nps}  (Δf ≈ {fs_k/nps:.1f} Hz, Δt ≈ {1000*nps/fs_k:.1f} ms)')
    ax[i].set_xlabel('Time (s)')
    ax[i].set_ylim(0, 5000)
    fig.colorbar(im, ax=ax[i], label='Magnitude')

ax[0].set_ylabel('Frequency (Hz)')
fig.suptitle('Knocking — STFT with varying nperseg (noverlap = nperseg/2)')
fig.tight_layout(); plt.show()

**관찰.** `nperseg=256` 은 각 노킹 충격 (세로 줄) 이 밀리초 단위로 날카롭게 찍히지만, 세로축 방향으로는 색이 넓게 번져 하모닉 구조가 거의 안 보인다. 반대로 `nperseg=8192` 는 가로 줄 (하모닉) 이 또렷하게 쌓이지만 개별 충격 시점은 좌우로 번져 식별하기 어렵다. 엔진음처럼 하모닉과 충격을 모두 관찰해야 하는 경우 `nperseg=2048` 부근이 실용적인 절충점이다.

---


## 실습 4. `noverlap` 효과 — 시간축 부드러움

`noverlap` 은 인접 세그먼트가 몇 샘플을 공유할지 결정한다. 중첩을 높이면 시간축 방향 프레임 수가 많아져 스펙트로그램이 부드럽게 보이지만, 계산량도 같은 비율로 증가한다.

`nperseg = 2048` 으로 고정하고 중첩 50 % / 75 % / 95 % 세 경우를 비교한다 (신호는 `knocking`).

In [ ]:
nperseg = 2048
overlap_ratios = [0.5, 0.75, 0.95]

fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, r in enumerate(overlap_ratios):
    nov = int(nperseg * r)
    f_k_i, t_k_i, Zxx_k_i = stft(v_k, fs=fs_k,
                                  nperseg=nperseg, noverlap=nov, nfft=nperseg)
    im = ax[i].pcolormesh(t_k_i, f_k_i, np.abs(Zxx_k_i),
                           shading='gouraud', cmap='viridis')
    ax[i].set_title(f'noverlap = {int(r*100)}%  ({len(t_k_i)} frames)')
    ax[i].set_xlabel('Time (s)')
    ax[i].set_ylim(0, 5000)
    fig.colorbar(im, ax=ax[i], label='Magnitude')

ax[0].set_ylabel('Frequency (Hz)')
fig.suptitle(f'Knocking — STFT with varying noverlap (nperseg={nperseg})')
fig.tight_layout(); plt.show()

**관찰.** 중첩을 높일수록 시간축 프레임 수가 크게 늘고 (제목에 표시), 스펙트로그램이 육안으로 부드럽게 이어진다. 특히 `knocking` 의 충격 간격이 95 % 중첩에서는 매끄러운 진행으로 보이는 반면, 50 % 중첩에서는 프레임 사이 계단이 느껴진다. 물리적인 정보량이 늘어난 것은 아니며, **시각적 보간** 에 가까운 효과다. 실무에서는 보통 50 ~ 75 % 를 표준으로 두고, 프레젠테이션용으로만 더 높인다.

---


## 실습 5. F1 엔진음 — STFT (`nperseg=2048`)

F1 엔진음은 회전수 (RPM) 가 수 초 안에 수만 RPM 까지 급격히 변한다. 이때는 하모닉 구조를 읽는 것이 중요하므로 **긴 윈도우 (주파수 해상도 우선)** 가 자연스럽다. `nperseg=2048` (≈ 46 ms), 50 % 중첩을 사용한다.

색상은 앞 두 신호와 구분되도록 `C2` (녹색) 를 쓴다.

In [ ]:
v_f1, fs_f1 = librosa.load('./data/f1music.mp3', sr=None)
t_f1_raw = np.arange(len(v_f1)) / fs_f1
print(f'F1 engine : shape={v_f1.shape}, fs={fs_f1} Hz, duration={len(v_f1)/fs_f1:.2f} s')

nperseg = 2048
noverlap = 1024
f_f1_stft, t_f1_stft, Zxx_f1 = stft(v_f1, fs=fs_f1,
                                     nperseg=nperseg, noverlap=noverlap)

fig, ax = plt.subplots(2, 1, figsize=(12, 8))

ax[0].plot(t_f1_raw, v_f1, 'C2')
ax[0].set_title('F1 Engine — Time')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')

im = ax[1].pcolormesh(t_f1_stft, f_f1_stft, np.abs(Zxx_f1),
                       shading='gouraud', cmap='viridis')
ax[1].set_title(f'F1 Engine — STFT (nperseg={nperseg}, noverlap={noverlap})')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Frequency (Hz)')
ax[1].set_ylim(0, 1000)
fig.colorbar(im, ax=ax[1], label='Magnitude')

fig.tight_layout(); plt.show()

**관찰.** 스펙트로그램의 하모닉 선이 시간 축을 따라 **위로 휘어 올라가는 곡선** 형태로 나타난다. 이는 RPM 상승에 따라 기본 주파수 `f0(t)` 와 그 정수배 `2f0, 3f0, …` 가 함께 올라가는 chirp 패턴이다. `nperseg=2048` 이 주는 약 22 Hz 의 주파수 해상도 덕분에 하모닉 사이 간격이 깨끗하게 분리된다. 시간축도 약 23 ms 프레임으로 충분히 촘촘해 RPM 변화를 추적할 수 있다.

---


## 실습 6. Mel Spectrogram — Linear vs dB

일반 STFT 는 주파수 축이 선형이고 크기 축도 선형이다. 반면 **청각 시스템**은

- 주파수를 **로그 스케일**로 인지한다 (저주파 영역의 100 Hz 차이가 고주파 영역의 1 kHz 차이보다 크게 느껴짐).
- 크기 역시 **로그 (dB) 스케일**로 인지한다.

이를 모사한 것이 `librosa.feature.melspectrogram` 이다. 같은 Mel 스펙트로그램을 크기만 바꿔 (linear / dB) 두 장으로 그려 비교한다. `win_length=2048`, `hop_length=1024` 로 앞 실습과 동일한 시간/주파수 그리드를 사용한다.

In [ ]:
win_length = 2048
hop_length = 1024

S = librosa.feature.melspectrogram(y=v_f1, sr=fs_f1,
                                    win_length=win_length,
                                    hop_length=hop_length)
S_dB = librosa.power_to_db(S, ref=np.max)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

img0 = librosa.display.specshow(S, x_axis='time', y_axis='mel',
                                 sr=fs_f1, hop_length=hop_length,
                                 cmap='magma', ax=ax[0])
ax[0].set_title('Mel Spectrogram — Linear')
ax[0].set_ylim(0, 1024)
fig.colorbar(img0, ax=ax[0], label='Magnitude (linear)')

img1 = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                                 sr=fs_f1, hop_length=hop_length,
                                 cmap='magma', ax=ax[1])
ax[1].set_title('Mel Spectrogram — dB (ref = max)')
ax[1].set_ylim(0, 1024)
fig.colorbar(img1, ax=ax[1], label='Magnitude (dB)')

fig.tight_layout(); plt.show()

**관찰.** 선형 스케일에서는 최고 크기 성분이 위치한 일부 셀만 밝게 타오르고, 나머지 대부분의 영역은 거의 검게 보여 구조를 읽기 어렵다. dB 로 바꾸면 동일 데이터의 **동적 범위**가 압축되어 배음·잔향·저레벨 노이즈 플로어까지 한 장에 드러난다. Mel 축 (로그 주파수) + dB 크기 조합이 음성·엔진음 시각화의 사실상 표준인 이유다.

---


## 요약

- STFT 는 **시간 × 주파수** 두 축에서 신호를 동시에 본다.
- `nperseg` 는 **시간 해상도 ↔ 주파수 해상도** 트레이드오프의 핵심 다이얼이며, 엔진음은 `2048` 이 실용적인 절충점이다.
- `noverlap` 은 시간축 **시각적 부드러움**을 좌우하나 실제 정보량은 늘리지 않는다. 표준은 50 ~ 75 %.
- 긴 윈도우 (주파수 해상도 우선) 는 하모닉·chirp 관찰에 적합하고, 짧은 윈도우 (시간 해상도 우선) 는 충격 이벤트 추적에 적합하다.
- 청각 해석이 필요한 경우 **Mel 축 + dB** 조합을 사용한다.
